In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


In [3]:
LOG_PATH = os.path.join(INGESTER_LOG_PATH, 'EmissionSourceSummaryIngester.log')
Logger = Loggers(logger_name = 'EmissionSourceSummaryIngester', keys = ['File', 'Slack'])
Logger.clear_handlers()
import logging

file_handler = logging.FileHandler(LOG_PATH)
# Set date format to dd-mm-yyyy in log output
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%d-%m-%Y')
file_handler.setFormatter(formatter)
Logger.File.addHandler(file_handler)

slack_handler = logging.StreamHandler(SlackWriter(channel = 'C0B9PGDNHH7'))
Logger.Slack.addHandler(slack_handler)

In [4]:
Logger.info("="*100)
Logger.Slack.info('Starting Emission Source Summary Ingester')                   

In [5]:
#Get customer list
customer_query = 'SELECT * FROM KPI_Customer'
customer_list = Query(query = "SELECT * FROM KPI_Customer WHERE DBLocation IS NOT 'Unknown'").execute(KPIHub_Conn)

#Set the time window for the update
current_date = date.today()
update_window = current_date - timedelta(days=UPDATE_WINDOW_DAYS)

In [6]:
def summarize_emission(group):
    # Remove all the rows in the group where Disposition == 2
    forCounts = group[group["Disposition"] != 2]
    forShares = group
    return pd.Series({
        "EmissionRate": forCounts["EmissionRate"].sum(),
        "LisaCount": forCounts["EmissionSourceId"].count(),
        "B0Count": forCounts["RepresentativeBinLabel"].value_counts().get("B0"),
        "B1Count": forCounts["RepresentativeBinLabel"].value_counts().get("B1"),
        "Bm1Count": forCounts["RepresentativeBinLabel"].value_counts().get("B-1"),
        "Bm2Count": forCounts["RepresentativeBinLabel"].value_counts().get("B-2"),
        "Not_NGCount": forShares["Disposition"].value_counts().get(2),
        "PGCount": forShares["Disposition"].value_counts().get(3),
        "NGCount": forShares["Disposition"].value_counts().get(1),
    })

In [8]:
#Fix fot no reports

for _,rows in customer_list.iterrows():
    customer_name = rows['Name']
    customer_id = rows['CustomerId']
    customer_db = rows['DBLocation']

    Logger.info(f"Processing customer: {customer_name}")
    Logger.info(f"Getting reports from {update_window} to {current_date}")
    #Query the last report
    reports = Query(
        f"""
        SELECT ReportId, ReportDate, LastUpdated FROM KPI_ReportSummary
        WHERE CustomerId = '{customer_id}'
        ORDER BY LastUpdated DESC
        """
    ).execute(KPIHub_Conn)

    emissions_count = Query(
        f"""
        SELECT COUNT(*) as EmissionCount
        FROM KPI_EmissionSourceSummary
        WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId = '{customer_id}')
        """
    ).execute(KPIHub_Conn)
    Logger.info(f"Getting emissions from {update_window} to {current_date}")

    if len(reports) > 0:
        if (emissions_count.iloc[0]['EmissionCount']) > 0:
            process_emissions = True
            starting_date = pd.to_datetime(update_window)

        else:
            Logger.info(f"No emissions found, processing")
            starting_date = datetime(STARTING_YEAR, 1, 1)
            process_emissions = True
    else:
        Logger.info(f"No reports found, skipping")
        process_emissions = False

    if process_emissions:
        # Ensure the ReportDate values are in datetime format before comparison,
        # handling both with and without microseconds (mixed formats)
        reports['ReportDate'] = pd.to_datetime(reports['ReportDate'], format='mixed')
   
        reports_to_query = reports[reports['ReportDate'] >= starting_date]
        reports_to_query.db.set_query(query_emission_sources_table(report_table = '#TempReports'))
        emission_sources = reports_to_query.db.execute(CONN_DICT[customer_db], source_col = 'ReportId', temp_table_name = '#TempReports')

        emissions_summary = emission_sources.groupby("ReportId").apply(summarize_emission).reset_index()
        emissions_summary.fillna(0, inplace=True)
        emissions_summary['LastUpdated'] = datetime.now()

        Logger.info(f"Reports from LSDB: {len(emissions_summary)}")
        KPI_EmissionSourceSummary.update_table(arguments = {'db_path': DB_PATH, 'DataFrame': emissions_summary, 'PrimaryKey': 'ReportId'})
        df_kpi = KPI_EmissionSourceSummary.query_table(arguments = {'db_path': DB_PATH})
        Logger.info(f"Reports from KPI_EmissionSourceSummary: {len(df_kpi)}")
    else:
        Logger.info(f"No reports found, skipping")



ProgrammingError: The second parameter to executemany must not be empty.